## Scraping AWOIAF Characters — v2 (Family-section excluded)

Same per-character scrape as `scrape_characters.ipynb`, but the `affiliated` column is rebuilt to **exclude links inside the `Family` h2 section** (which on AWOIAF wraps `Ancestors`, `Descendants`, family trees, etc.). v1's affiliated list walked the whole article body, so it bundled multi-century genealogy with actual storyline links — producing spurious edges between e.g. Daenerys and ancient Targaryen kings.

Reuses the existing `characters.csv` (the wiki's character roster hasn't changed); only re-scrapes the per-character pages. Output: `characters_enriched_v2.csv`.

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor


### Setup
Same browser User-Agent, session, and infobox label map as v1 — the wiki returns 403 for generic bot UAs, and accepts both `Allegiance`/`Allegiances` plus `Lover`/`Lovers`/`Paramour` variants.

In [ ]:
BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

INFOBOX_LABELS = {
    'Father': 'father', 'Fathers': 'father',
    'Mother': 'mother', 'Mothers': 'mother',
    'Spouse': 'spouse', 'Spouses': 'spouse',
    'Lover': 'lover', 'Lovers': 'lover', 'Paramour': 'lover',
    'Issue': 'issue', 'Issues': 'issue',
    'Allegiance': 'allegiance', 'Allegiances': 'allegiance',
}

session = requests.Session()
session.headers.update(headers)


### Helpers
`slug_to_url`, `href_to_slug`, `cell_links_or_text`, `parse_infobox` are identical to v1. The only new piece is `parse_affiliated_no_family`.

In [ ]:
def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def href_to_slug(href):
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):].split('#', 1)[0]
    if not slug or slug.startswith('Special:'):
        return None
    return unquote(slug)


def cell_links_or_text(td):
    slugs = []
    seen = set()
    for a in td.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    if slugs:
        return ';'.join(slugs)
    text = td.get_text(' ', strip=True)
    return text if text else ''


def parse_infobox(soup):
    data = {col: '' for col in INFOBOX_LABELS.values()}
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        return data
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        label = th.get_text(strip=True).rstrip(':')
        col = INFOBOX_LABELS.get(label)
        if col is None:
            continue
        if data[col] == '':
            data[col] = cell_links_or_text(td)
    return data


### New `parse_affiliated_no_family`
Walks the article body for internal `/index.php/` links like v1, but first builds a set of every DOM node that lives **inside any `Family` h2 section** (the heading itself, its descendants, every sibling up to the next h2, and their descendants). Links whose own `id()` is in that set are skipped.

Matching is case-insensitive — the survey showed `Family` is consistent, but the same pattern lets you add more sections later by extending `EXCLUDED_H2_SECTIONS`.

In [ ]:
EXCLUDED_H2_SECTIONS = {'family'}  # lowercased; extend if you want to drop more


def parse_affiliated_no_family(soup, self_slug):
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''

    excluded = set()
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if not span:
            continue
        if span.get_text(strip=True).lower() not in EXCLUDED_H2_SECTIONS:
            continue
        excluded.add(id(h))
        for d in h.find_all(True):
            excluded.add(id(d))
        for sib in h.find_next_siblings():
            if sib.name == 'h2':
                break
            excluded.add(id(sib))
            for d in sib.find_all(True):
                excluded.add(id(d))

    slugs = []
    seen = set()
    for a in content.find_all('a'):
        if id(a) in excluded:
            continue
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug == self_slug or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    return ';'.join(slugs)


### Per-character scrape
Same shape as v1's `scrape_character`, only the affiliated parser changes.

In [ ]:
EMPTY = {'father': '', 'mother': '', 'spouse': '', 'lover': '', 'issue': '',
         'allegiance': '', 'affiliated': ''}


def scrape_character(row):
    name, slug = row['name'], row['ID']
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return {'name': name, 'ID': slug, **EMPTY}
    if resp.status_code != 200:
        print(f'  ! {resp.status_code} for {slug}')
        return {'name': name, 'ID': slug, **EMPTY}
    soup = BeautifulSoup(resp.content, 'html.parser')
    info = parse_infobox(soup)
    return {
        'name': name,
        'ID': slug,
        'father': info['father'],
        'mother': info['mother'],
        'spouse': info['spouse'],
        'lover': info['lover'],
        'issue': info['issue'],
        'allegiance': info['allegiance'],
        'affiliated': parse_affiliated_no_family(soup, slug),
    }


### Run the scrape
~3,689 characters, 8 threads, checkpointed every 100 rows. Same runtime as v1.

In [ ]:
characters = pd.read_csv('../csvs/characters.csv').to_dict('records')
print(f'{len(characters)} characters to enrich')

COLUMNS = ['name', 'ID', 'father', 'mother', 'spouse', 'lover', 'issue', 'allegiance', 'affiliated']
OUT = 'characters_enriched_v2.csv'
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for i, row in enumerate(executor.map(scrape_character, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'Saved {len(rows)} rows to {OUT}')


### Restrict slug-valued columns to characters only
Identical to v1 section 3: drop non-character internal links from `father, mother, spouse, lover, issue, affiliated`, and free-text fallbacks from `allegiance`.

In [ ]:
df = pd.read_csv(OUT).fillna('')
character_ids = set(df['ID'])

SLUG_COLUMNS = ['father', 'mother', 'spouse', 'lover', 'issue', 'affiliated']


def keep_characters(cell):
    if not cell:
        return ''
    return ';'.join(s for s in cell.split(';') if s in character_ids)


def count_entries(col):
    return df[col].str.count(';').add(df[col].ne('').astype(int)).sum()


for col in SLUG_COLUMNS:
    before = count_entries(col)
    df[col] = df[col].map(keep_characters)
    after = count_entries(col)
    print(f'  {col:11s}: {before:>6} -> {after:<6}  (dropped {before - after})')

before = count_entries('allegiance')
df['allegiance'] = df['allegiance'].map(
    lambda c: ';'.join(s for s in c.split(';') if s and ' ' not in s and '[' not in s) if c else ''
)
after = count_entries('allegiance')
print(f'  {"allegiance":11s}: {before:>6} -> {after:<6}  (dropped {before - after} free-text entries)')

df.to_csv(OUT, index=False)


### Diff vs. v1
Per-character count of how many affiliated links got dropped by skipping the Family section. The mean delta is the average genealogy-noise edge count per character; the top-10 list shows where Family was most polluting (expect Targaryens, Starks, and other dynastic figureheads).

In [ ]:
v1 = pd.read_csv('../csvs/characters_enriched.csv').fillna('')
v2 = pd.read_csv(OUT).fillna('')

merged = v1[['ID', 'affiliated']].merge(v2[['ID', 'affiliated']], on='ID', suffixes=('_v1', '_v2'))


def count_links(s):
    return len([x for x in s.split(';') if x])


merged['n_v1'] = merged['affiliated_v1'].map(count_links)
merged['n_v2'] = merged['affiliated_v2'].map(count_links)
merged['dropped'] = merged['n_v1'] - merged['n_v2']

print(f'Total affiliated links:  v1 = {merged["n_v1"].sum():,}   v2 = {merged["n_v2"].sum():,}')
print(f'Mean per character:      v1 = {merged["n_v1"].mean():.1f}   v2 = {merged["n_v2"].mean():.1f}')
print(f'Mean dropped:            {merged["dropped"].mean():.1f}')
print(f'Characters with zero drops: {(merged["dropped"] == 0).sum()} / {len(merged)}')
print()
print('Top 15 characters by Family-section link drop:')
print(merged.nlargest(15, 'dropped')[['ID', 'n_v1', 'n_v2', 'dropped']].to_string(index=False))
